# End-to-End Tabular Feature Engineering & Predictive Modeling Pipeline
## Feature Selection via Linear Discriminant Analysis (LDA) & Gradient Boosting Regression
**Author:** Eugene Phang  
**Domain:** Predictive Analytics, Feature Selection, Tabular Machine Learning  

---

### Overview
This notebook demonstrates an end-to-end predictive modeling workflow for competitive sports/race outcome forecasting. Key stages of the pipeline include:
1. **Data Ingestion & Alignment:** Multi-source file extraction, date parsing, and relational joining across race participants and historical form metrics.
2. **Domain-Specific Feature Engineering:** Categorical encoding, historical win/place ratios, track conditions, and sequential features.
3. **Dimensionality Reduction & Feature Selection:** Utilizing Linear Discriminant Analysis (LDA) component coefficients (LD1/LD2) to select the most discriminative subset of features.
4. **Cross-Validated Regression:** Model training with Gradient Boosting Regressors evaluated under 5-Fold cross-validation, hyperparameter tuning with , and feature importance attribution.


In [278]:
from google.colab import drive
drive.mount('/content/drive')
import os
#os.chdir('c:/py/data')
os.chdir('/content/drive/MyDrive/repo/Lexi') #<-- your local
base_dir = 'research/2026/01/'
print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/repo/Lexi


In [279]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

File extraction - by track

In [280]:
def extract_date_from_filename(filename):
        # Extract date pattern like 2026-01-07 or 260107
        date_match = re.search(r'(\d{4}-\d{2}-\d{2})|(\d{6})', filename)
        if date_match:
            date_str = date_match.group(0)
            if '-' in date_str:
                # Format: YYYY-MM-DD
                return date_str
            else:
                # Format: YYMMDD
                year = '20' + date_str[:2]
                month = date_str[2:4]
                day = date_str[4:6]
                return f"{year}-{month}-{day}"
        return "0000-00-00"  # Default for files without dates


"""Get all Eagle Farm form and results files, sorted by date in filename"""
# Find all form and results files
form_files = []
results_files = []

for filename in os.listdir(base_dir):
    if 'form-data' in filename and 'eagle-farm' in filename:
        form_files.append(filename)
    elif 'results-data' in filename and 'eagle-farm' in filename:
        results_files.append(filename)

# Sort files by date in filename (extract date from filename)

# Extract date pattern like 2026-01-07 or 260107
#filename = 'form-data-2026-01-28-eagle-farm.csv'



form_files.sort(key=extract_date_from_filename)
results_files.sort(key=extract_date_from_filename)
print(f"Found {len(form_files)} form files: {form_files}")
print(f"Found {len(results_files)} results files: {results_files}")

Found 2 form files: ['form-data-2026-01-28-eagle-farm.csv', 'form-data-2026-01-31-eagle-farm.csv']
Found 1 results files: ['results-data-2026-01-28-eagle-farm.csv']


Merge into one dataframe

In [282]:
# Load and concatenate form data
form_data_list = []
for form_file in form_files:
    filepath = f'research/2026/01/{form_file}'
    try:
        df = pd.read_csv(filepath)
        print(f"Loaded form data from {form_file} with shape: {df.shape}")
        form_data_list.append(df)
    except FileNotFoundError:
        print(f"Warning: Form file {filepath} not found")

# Load and concatenate results data
results_data_list = []
for results_file in results_files:
    filepath = f'research/2026/01/{results_file}'
    try:
        df = pd.read_csv(filepath)
        print(f"Loaded results data from {results_file} with shape: {df.shape}")
        results_data_list.append(df)
    except FileNotFoundError:
        print(f"Warning: Results file {filepath} not found")

if form_data_list:
    all_form_data = pd.concat(form_data_list, ignore_index=True)
    print(f"Combined form data shape: {all_form_data.shape}")
else:
    print("No form data files found.")


if results_data_list:
    all_results_data = pd.concat(results_data_list, ignore_index=True)
    print(f"Combined results data shape: {all_results_data.shape}")
else:
    print("No results data files found.")

Loaded form data from form-data-2026-01-28-eagle-farm.csv with shape: (71, 781)
Loaded form data from form-data-2026-01-31-eagle-farm.csv with shape: (118, 781)
Loaded results data from results-data-2026-01-28-eagle-farm.csv with shape: (71, 23)
Combined form data shape: (189, 781)
Combined results data shape: (71, 23)


Merge by Horse Name

In [283]:
form_data = all_form_data
results_data = all_results_data


"""Merge form and results data and create engineered features"""
# Normalize horse names for merging
form_data['horse_normalized'] = form_data['Name'].str.lower().str.replace(' ', '').str.replace("'", "")
results_data['horse_normalized'] = results_data['horse'].str.lower().str.replace(' ', '').str.replace("'", "")

# Merge datasets
merged_data = pd.merge(
    form_data,
    #results_data[['horse_normalized', 'position', 'race number', 'margin', 'price sp']],
    results_data,
    on='horse_normalized',
    how='inner'
)

print(f"Merged dataset shape: {merged_data.shape}")
#print(f"Unique races: {merged_data['race number'].nunique()}")
#print(f"Positions range: {merged_data['position'].min()} to {merged_data['position'].max()}")

Merged dataset shape: (72, 805)


Full list of features

In [ ]:
#print('All features in the merged dataset:')
#display(merged_data.columns.tolist())

Many strings should turned into categories

In [284]:
#TODO Domain
# Encode categorical variables
categorical_cols = ['Trainer.FullName', 'Jockey.FullName', 'Sex']
for col in categorical_cols:
    if col in merged_data.columns:
        le = LabelEncoder()
        # Handle unknown categories by fitting on all data first
        le.fit(merged_data[col].astype(str))
        merged_data[f'{col}_encoded'] = le.transform(merged_data[col].astype(str))

**Feature Merge TODO**
Data that appears multiple times might need special attention

This will be important when data comes in as a sequence right up to critical moments. eg. patterns per horse leading up to the race. As horse data gets updated the model should be triggered to make new predictions.

These merging algorithms don't do anything for now. I've left them here to go through later.
```

# This is formatted as code
# Feature engineering
if 'WinPct' in merged_data.columns:
    merged_data['combined_skill'] = merged_data['WinPct'] * merged_data['PlacePct']
else:
    merged_data['combined_skill'] = np.random.beta(2, 3, len(merged_data))

if 'GoodRecord.Firsts' in merged_data.columns:
    merged_data['experience_score'] = (
        merged_data['GoodRecord.Firsts'] +
        merged_data['GoodRecord.Seconds'] +
        merged_data['GoodRecord.Thirds']
    ) / 3
else:
    merged_data['experience_score'] = np.random.poisson(2, len(merged_data))

if 'Weight' in merged_data.columns and 'DistanceRecord.Firsts' in merged_data.columns:
    merged_data['form_efficiency'] = merged_data['DistanceRecord.Firsts'] / (merged_data['Weight'] + 1)
else:
    merged_data['form_efficiency'] = np.random.normal(0.1, 0.05, len(merged_data))

if 'Barrier' in merged_data.columns:
    merged_data['barrier_advantage'] = 8.5 - abs(merged_data['Barrier'] - 8.5)
else:
    merged_data['barrier_advantage'] = np.random.uniform(-3, 3, len(merged_data))
```



Features selected via Linear Discriminant Analysis -see below (formerly handpicked with Orange3 workflow)

In [285]:
# Define potential feature columns based on typical form data
potential_features_from_ows = [
    'Age', 'Weight', 'Barrier', 'TabNo', 'PrizeMoney',
    'CareerStarts', 'CareerWins', 'CareerSeconds', 'CareerThirds',
    'WinPct', 'PlacePct', 'combined_skill', 'experience_score',
    'form_efficiency', 'barrier_advantage', 'price sp',
    'Trainer.FullName_encoded', 'Jockey.FullName_encoded', 'Sex_encoded',
    'GoodRecord.Firsts', 'GoodRecord.Seconds', 'GoodRecord.Thirds',
    'DistanceRecord.Firsts', 'TrackRecord.Firsts'
]


#Linear Discriminant features LD1:
potential_features = ['CareerStarts', 'Age', 'CareerWins', 'WinPct', 'GoodRecord.Firsts', 'GoodRecord.Thirds', 'PlacePct', 'Trainer.FullName_encoded', 'TabNo', 'GoodRecord.Seconds', 'CareerThirds', 'DistanceRecord.Firsts', 'CareerSeconds', 'Jockey.FullName_encoded', 'TrackRecord.Firsts', 'Sex_encoded', 'Barrier', 'Weight', 'price sp', 'PrizeMoney']

#Linear Discriminant features combined LD1 and LD2:
#potential_features = ['CareerWins', 'CareerStarts', 'GoodRecord.Seconds', 'WinPct', 'GoodRecord.Firsts', 'PlacePct', 'Age', 'PrizeMoney', 'CareerThirds', 'GoodRecord.Thirds', 'TabNo', 'DistanceRecord.Firsts', 'CareerSeconds', 'Barrier', 'TrackRecord.Firsts', 'Weight', 'price sp']

# Filter to only available columns
feature_columns = [col for col in potential_features if col in merged_data.columns]

#print(f"Selected {len(feature_columns)} features for modeling: {feature_columns}")

X = merged_data[feature_columns]
y_position = merged_data['position']

feature_columns
#X
#y_position

['CareerStarts',
 'Age',
 'CareerWins',
 'WinPct',
 'GoodRecord.Firsts',
 'GoodRecord.Thirds',
 'PlacePct',
 'Trainer.FullName_encoded',
 'TabNo',
 'GoodRecord.Seconds',
 'CareerThirds',
 'DistanceRecord.Firsts',
 'CareerSeconds',
 'Jockey.FullName_encoded',
 'TrackRecord.Firsts',
 'Sex_encoded',
 'Barrier',
 'Weight',
 'price sp',
 'PrizeMoney']

LDA stuff should be done after standardising X - but the Gradient Boost Regressor doesn't need standardised numbers.


In [286]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd
import numpy as np

"""Handle missing values and standardize features"""
# Handle missing values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

# Standardize features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X_imputed.columns)

# Apply Linear Discriminant Analysis (LDA)
# LDA requires the target variable to be discrete classes.
# Here, 'position' is treated as discrete classes for dimensionality reduction.
lda = LinearDiscriminantAnalysis(n_components=min(X_scaled.shape[1], y_position.nunique() - 1))
X_lda = lda.fit_transform(X_scaled, y_position)

#print("LDA Explained Variance Ratio:")
#display(pd.DataFrame(lda.explained_variance_ratio_, index=[f'LD{i+1}' for i in range(len(lda.explained_variance_ratio_))], columns=['Explained Variance Ratio']))

print("\nLDA Coefficients (contribution of each feature to discriminant functions):")
display(pd.DataFrame(lda.coef_, columns=X_scaled.columns, index=[f'LD{i+1}' for i in range(len(lda.coef_))]))


LDA Coefficients (contribution of each feature to discriminant functions):


,CareerStarts,Age,CareerWins,WinPct,GoodRecord.Firsts,GoodRecord.Thirds,PlacePct,Trainer.FullName_encoded,TabNo,GoodRecord.Seconds,CareerThirds,DistanceRecord.Firsts,CareerSeconds,Jockey.FullName_encoded,TrackRecord.Firsts,Sex_encoded,Barrier,Weight,price sp,PrizeMoney
LD1,2.424448,-3.047224,-2.657334,1.064420,1.206946,-0.618453,0.764295,-1.094559,-0.684569,0.147194,-0.537410,0.954099,-0.736523,0.194053,0.026317,0.135937,0.056349,0.038831,0.214862,2.753970
LD2,2.588078,-2.945915,-0.733826,-0.555257,0.406777,0.316806,1.707723,0.276250,-0.959435,1.576942,-2.154839,0.111505,0.883912,0.293323,-0.165577,0.413657,-0.086254,-0.374865,-0.054224,-0.519076
LD3,-3.598854,0.317891,-0.025407,-0.583049,0.699058,1.787625,1.810881,0.063160,-1.009600,-0.000282,-0.242319,-0.766773,0.742930,0.793299,1.123208,0.629996,0.330810,0.260698,1.140886,0.633094
LD4,3.652090,-3.026900,-0.892674,0.635411,-0.372816,-0.386645,-0.625494,-0.596080,-0.604213,-0.542206,0.255612,2.059979,-2.327836,0.106572,0.409679,-0.938686,-0.148716,-1.589091,-0.113836,1.595272
LD5,2.627953,1.580561,-1.737661,-0.407815,0.391010,-1.529826,-0.214263,1.408798,0.651214,-0.047284,0.058251,0.339368,-0.751980,-0.092676,0.046706,-0.157050,-1.117216,-0.472697,-1.166106,-0.773301
LD6,-1.918984,2.785473,2.752569,-0.747526,-0.230599,0.101150,-0.844392,0.398861,0.819415,0.975370,0.543842,-1.742289,1.060753,-0.482690,-0.768826,-0.269074,-0.556160,0.877506,-0.804311,-3.291026
LD7,-2.134552,2.233688,2.980664,-0.511177,-0.461406,0.390029,-1.529718,-0.487940,0.133267,-1.422941,1.057188,-1.233594,0.885421,-0.540646,-0.782603,-0.063835,0.164119,0.648657,-0.840874,-1.762067
LD8,-3.647102,0.746702,0.216491,0.847967,-0.753206,0.286098,-0.444516,0.142319,1.082488,-0.694869,1.316572,0.887123,-0.196970,0.242798,0.833320,-0.181266,1.049437,0.080678,2.435814,1.829734
LD9,-0.580182,2.047494,-0.061540,-1.369008,0.208604,-0.503837,0.013611,0.901043,0.633210,-0.331112,0.196579,-0.976420,0.605667,0.017015,-0.632743,0.891512,0.106765,0.799145,-0.816458,-0.593532
LD10,-3.307623,4.350252,4.006270,0.379040,-2.536796,0.133030,-1.989031,-0.854773,0.877105,0.026740,-0.175799,-1.750569,1.090016,-1.677723,-1.139672,0.208161,0.477635,0.957787,-1.141272,-2.607885


Option to use combined LD1 and LD2 coefficients can be found where potential_features is defined. I've use just LD1 for now. LD2 has less features.

In [287]:
if hasattr(lda, 'coef_'):
    # Combine LD1 and LD2 coefficients, taking the maximum absolute value for each feature
    if not ld2_coefficients.empty:
        combined_coefficients = pd.concat([ld1_coefficients, ld2_coefficients], axis=1).max(axis=1)
    else:
        combined_coefficients = ld1_coefficients

    # Sort the combined coefficients in descending order
    combined_coefficients = combined_coefficients.sort_values(ascending=False)

    print("Prioritized features based on highest absolute coefficient across LD1 and LD2:")
    display(combined_coefficients)
    print("\nList of features in the combined coefficients:")
    print(combined_coefficients.index.tolist())
else:
    print("LDA coefficients not available. Ensure LDA was fitted correctly.")

Prioritized features based on highest absolute coefficient across LD1 and LD2:


,0
Age,3.047224
PrizeMoney,2.753970
CareerWins,2.657334
CareerStarts,2.588078
CareerThirds,2.154839
PlacePct,1.707723
GoodRecord.Seconds,1.576942
GoodRecord.Firsts,1.206946
Trainer.FullName_encoded,1.094559
WinPct,1.064420



List of features in the combined coefficients:
['Age', 'PrizeMoney', 'CareerWins', 'CareerStarts', 'CareerThirds', 'PlacePct', 'GoodRecord.Seconds', 'GoodRecord.Firsts', 'Trainer.FullName_encoded', 'WinPct', 'TabNo', 'DistanceRecord.Firsts', 'CareerSeconds', 'GoodRecord.Thirds', 'Sex_encoded', 'Weight', 'Jockey.FullName_encoded', 'price sp', 'TrackRecord.Firsts', 'Barrier']


In [288]:
if hasattr(lda, 'coef_'):
    # 1. Extract the coefficients for the first linear discriminant (LD1)
    # 2. Create a Pandas Series named ld1_coefficients using these coefficients and X_scaled.columns as the index. Calculate the absolute value for each coefficient.
    ld1_coefficients = pd.Series(lda.coef_[0], index=X_scaled.columns).abs()

    # 3. Extract the coefficients for the second linear discriminant (LD2)
    # 4. Create a Pandas Series named ld2_coefficients using these coefficients and X_scaled.columns as the index. Calculate the absolute value for each coefficient.
    if lda.n_components > 1:
        ld2_coefficients = pd.Series(lda.coef_[1], index=X_scaled.columns).abs()
    else:
        ld2_coefficients = pd.Series([], dtype='float64') # Empty series if only 1 component

    # 5. Sort both ld1_coefficients and ld2_coefficients in descending order based on their absolute values.
    ld1_coefficients = ld1_coefficients.sort_values(ascending=False)
    ld2_coefficients = ld2_coefficients.sort_values(ascending=False)

    print("Top features for LD1 (absolute coefficients):")
    display(ld1_coefficients)

    if not ld2_coefficients.empty:
        print("\nTop features for LD2 (absolute coefficients):")
        display(ld2_coefficients)
    else:
        print("\nOnly one LDA component was found, so LD2 coefficients are not available.")
else:
    print("LDA coefficients not available. Ensure LDA was fitted correctly.")

Top features for LD1 (absolute coefficients):


,0
Age,3.047224
PrizeMoney,2.753970
CareerWins,2.657334
CareerStarts,2.424448
GoodRecord.Firsts,1.206946
Trainer.FullName_encoded,1.094559
WinPct,1.064420
DistanceRecord.Firsts,0.954099
PlacePct,0.764295
CareerSeconds,0.736523



Top features for LD2 (absolute coefficients):


,0
Age,2.945915
CareerStarts,2.588078
CareerThirds,2.154839
PlacePct,1.707723
GoodRecord.Seconds,1.576942
TabNo,0.959435
CareerSeconds,0.883912
CareerWins,0.733826
WinPct,0.555257
PrizeMoney,0.519076


Model training here - 5-fold splits for cross validation - so the internal predictability is measured five times against randomly shuffled partitions of 5 to 1 training to test.

See 5 readings for the regression accuracy and an average across each result.



In [289]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
gb_regressor = GradientBoostingRegressor(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=1,
    min_samples_split=5,
    random_state=42
)

"""Perform k-fold cross-validation for regression"""
kf = KFold(n_splits=5, shuffle=True, random_state=42)
Xr = X #n: use X_standardised for non tree algorithms
yr = y_position
# Define the model

model = gb_regressor
model.fit(Xr, yr)
# Perform cross-validation for different metrics
mse_scores = cross_val_score(model, Xr, yr, cv=kf, scoring='neg_mean_squared_error')
mae_scores = cross_val_score(model, Xr, yr, cv=kf, scoring='neg_mean_absolute_error')
r2_scores = cross_val_score(model, Xr, yr, cv=kf, scoring='r2')

# Convert negative scores to positive
mse_scores = -mse_scores
mae_scores = -mae_scores

# Calculate RMSE scores
rmse_scores = np.sqrt(mse_scores)

# Calculate average MAE
mean_mae = np.mean(mae_scores)

display({
    'rmse': rmse_scores,
    'rmse_average': round(np.mean(rmse_scores), 2),
    'mae' : mae_scores,
    'mae_average': round(mean_mae, 2),
    'r2': r2_scores,
    'r2_average':  round(np.mean(r2_scores), 2)
})

{'rmse': array([2.04928166, 1.86813039, 2.22381031, 2.72036585, 2.66250232]),
 'rmse_average': np.float64(2.3),
 'mae': array([1.63912538, 1.50425923, 1.95865837, 2.12354131, 2.12999192]),
 'mae_average': np.float64(1.87),
 'r2': array([0.24769112, 0.53972449, 0.40752743, 0.10684944, 0.12061516]),
 'r2_average': np.float64(0.28)}

Table of horse position predictions are pretty good. For only 2 days worth of data.

In [290]:
"""Generate detailed report with predicted and actual positions for all horses"""
# Make predictions on the full dataset

# Fit the model on the entire dataset before making predictions

all_pos_predictions = model.predict(X)
# Create detailed report with predicted positions rounded to 0 decimal places
detailed_report = pd.DataFrame({
    'Horse_Name': merged_data.get('Name', [f'Horse_{i}' for i in range(len(merged_data))]),
    'Actual_Position': y_position.values,
    'Predicted_Position': np.round(all_pos_predictions, 0).astype(int),
    'Position_Error': np.abs(y_position.values - np.round(all_pos_predictions, 0).astype(int))
})

display(detailed_report)

,Horse_Name,Actual_Position,Predicted_Position,Position_Error
0,Concador,7,6,1
1,Montevecchio,2,5,3
2,Sowilo Rodina,3,5,2
3,Enjector,6,5,1
4,Lottery Link,9,6,3
...,...,...,...,...
67,First Empire,2,2,0
68,Sexy Time,3,5,2
69,Tap High,4,5,1
70,Espoir Da Tap,11,7,4


I guess it's not surprising that starting price accounts for the largest bulk of helpful numbers. 40% it looks like. The rest are pretty trace comparitively.


In [292]:
"""Analyze and return feature importances"""
reg_model = model #<-- gb_regressor
feature_columns = X.columns
reg_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': reg_model.feature_importances_
}).sort_values('importance', ascending=False)

reg_importance

,feature,importance
18,price sp,0.407255
13,Jockey.FullName_encoded,0.091309
6,PlacePct,0.085676
8,TabNo,0.085497
16,Barrier,0.079676
9,GoodRecord.Seconds,0.076538
4,GoodRecord.Firsts,0.053260
7,Trainer.FullName_encoded,0.043603
19,PrizeMoney,0.039094
17,Weight,0.029376


This is a method for automatically finding the best parameters for the Regressor. The lowest Mean Absolute Error using this method is 1.87 positions from true values.



In [ ]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

gb_regressor = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=1,
    min_samples_split=2,
    #random_state=42
)

"""Perform k-fold cross-validation for regression"""
kf = KFold(n_splits=5, shuffle=True, random_state=42)
Xr = X #n: use X_standardised for non tree algorithms
yr = y_position

# Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [1, 2, 3],
    'min_samples_split': [2, 5]
}

# Instantiate GridSearchCV
grid_search = GridSearchCV(
    estimator=gb_regressor,
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_squared_error',
    n_jobs=-1, # Use all available cores
    verbose=0
)

# Fit GridSearchCV to the data
grid_search.fit(Xr, yr)

# Update the model with the best estimator found by GridSearchCV
model = grid_search.best_estimator_

# Print the best parameters
print("Best parameters found by GridSearchCV:")
display(grid_search.best_params_)

# Perform cross-validation for different metrics using the best model
mse_scores = cross_val_score(model, Xr, yr, cv=kf, scoring='neg_mean_squared_error')
mae_scores = cross_val_score(model, Xr, yr, cv=kf, scoring='neg_mean_absolute_error')
r2_scores = cross_val_score(model, Xr, yr, cv=kf, scoring='r2')

# Convert negative scores to positive
mse_scores = -mse_scores
mae_scores = -mae_scores

# Calculate RMSE scores
rmse_scores = np.sqrt(mse_scores)

# Calculate average MAE
mean_mae = np.mean(mae_scores)

display({
    'rmse': rmse_scores,
    'rmse_average': round(np.mean(rmse_scores), 2),
    'mae' : mae_scores,
    'mae_average': round(mean_mae, 2),
    'r2': r2_scores,
    'r2_average':  round(np.mean(r2_scores), 2)
})

Best parameters found by GridSearchCV:


{'learning_rate': 0.05,
 'max_depth': 1,
 'min_samples_split': 2,
 'n_estimators': 150}

{'rmse': array([2.04928166, 1.86813039, 2.22381031, 2.72036585, 2.66250232]),
 'rmse_average': np.float64(2.3),
 'mae': array([1.63912538, 1.50425923, 1.95865837, 2.12354131, 2.12999192]),
 'mae_average': np.float64(1.87),
 'r2': array([0.24769112, 0.53972449, 0.40752743, 0.10684944, 0.12061516]),
 'r2_average': np.float64(0.28)}